## PYMUPDF

In [ ]:
!pip install pymupdf pdfplumber pdfminer.six

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## PDFPlumber

In [1]:
import pdfplumber

pdf_path = "..\\Sample PDFs\\InfoEdge_Annual_Report_2025.pdf"
output_path = "plumber_output.txt"

with pdfplumber.open(pdf_path) as pdf, open(output_path, "w", encoding="utf-8") as f:
    for i, page in enumerate(pdf.pages):
        text = page.extract_text() or ""
        f.write(f"\n--- Page {i+1} ---\n")
        f.write(text)

## PyMuPDF

In [9]:
import fitz  # PyMuPDF

pdf_path = "..\\Sample PDFs\\Form.pdf"
output_path = "pymupdf_output.txt"

doc = fitz.open(pdf_path)

with open(output_path, "w", encoding="utf-8") as f:
    for i, page in enumerate(doc):
        text = page.get_text()
        f.write(f"\n--- Page {i+1} ---\n")
        f.write(text)

## PDFMiner

In [10]:
from pdfminer.high_level import extract_text

pdf_path = "..\\Sample PDFs\\Form.pdf"
output_path = "pdfminer_output.txt"

text = extract_text(pdf_path)

with open(output_path, "w", encoding="utf-8") as f:
    f.write(text)

## PDFMiner Text Annotations

In [12]:
from pathlib import Path
from pdfminer.high_level import extract_pages
from pdfminer.layout import LTTextContainer
import fitz  # PyMuPDF

pdf_path = "..\\Sample PDFs\\Form.pdf"
output_dir = Path("output_3")
output_dir.mkdir(exist_ok=True)

doc = fitz.open(pdf_path)
counter = 1

zoom = 2.0
matrix = fitz.Matrix(zoom, zoom)

for page_num, layout in enumerate(extract_pages(pdf_path), start=1):
    page = doc[page_num - 1]
    h = page.rect.height
    shape = page.new_shape()

    for element in layout:
        if isinstance(element, LTTextContainer):
            x0, y0, x1, y1 = element.bbox
            rect = fitz.Rect(x0, h - y1, x1, h - y0)

            shape.draw_rect(rect)
            shape.insert_text((rect.x0, rect.y0 - 2), str(counter), fontsize=8)
            counter += 1

    shape.finish(color=(1, 0, 0), width=1)
    shape.commit()

    pix = page.get_pixmap(matrix=matrix)
    pix.save(output_dir / f"page_{page_num}.png")

doc.close()